In [1]:
from typing import Any
from typing import Dict

import gymnasium as gym
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.env_util import make_vec_env
import numpy as np

ENV_ID = "SabreSwapEnv"

gym.register(
    id=ENV_ID, entry_point="utils.sabre_env_wog_wc:SabreSwapEnv"
)

N_TRIALS = 2000
N_STARTUP_TRIALS = 10
N_EVALUATIONS = 5
N_TIMESTEPS = int(1e6)
EVAL_FREQ = int(N_TIMESTEPS / N_EVALUATIONS)
N_EVAL_EPISODES = 5

DEFAULT_HYPERPARAMS = {
    "policy": "MlpPolicy",
    "env": make_vec_env(ENV_ID, n_envs=4, env_kwargs={"start_level": 1})
}


d:\lab\circuit_route\.venv\Lib\site-packages\gymnasium\envs\registration.py:736: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes ([]).
  logger.warn(


In [2]:
def sample_ppo_params(trial: optuna.Trial) -> Dict[str, Any]:
    """Sampler for PPO hyperparameters."""
    gamma: float = 1.0 - trial.suggest_float("gamma", 0.0001, 0.1, log=True)
    max_grad_norm: float = trial.suggest_float("max_grad_norm", 0.3, 5.0, log=True)
    gae_lambda: float = 1.0 - trial.suggest_float("gae_lambda", 0.001, 0.1, log=True)
    n_steps: int = 2 ** trial.suggest_int("exponent_n_steps", 3, 11)
    learning_rate: float = trial.suggest_float("lr", 1e-5, 1, log=True)
    ent_coef: float = trial.suggest_float("ent_coef", 0.0000001, 0.1, log=True)
    vf_coef: float = trial.suggest_float("vf_coef", 0.000001, 1.0, log=True)

    # Display true values.
    trial.set_user_attr("gamma_", gamma)
    trial.set_user_attr("gae_lambda_", gae_lambda)
    trial.set_user_attr("n_steps", n_steps)


    return {
        "n_steps": n_steps,
        "gamma": gamma,
        "gae_lambda": gae_lambda,
        "learning_rate": learning_rate,
        "ent_coef": ent_coef,
        "max_grad_norm": max_grad_norm,
        "vf_coef": vf_coef,
    }

In [3]:
class CurriculumCallback(BaseCallback):
    """Optimized callback for curriculum learning."""

    def __init__(self, max_level: int = 1000, min_training_epi: int = 1000 ,success_threshold: float = 0.8, verbose: int = 0):
        super().__init__(verbose)
        self.max_level = max_level
        self.success_threshold = success_threshold
        self.min_training_epi = min_training_epi
        
        self.level = 1
        self.current_success_rate = 0.0

        # 로깅 주기 설정 (매 스텝이 아닌 주기적으로)
        self.log_freq = 100
        self.step_count = 0
        self.episode_count = 0
        self.truncated_or_success_count = 0

    def _on_step(self) -> bool:
        dones = self.locals.get("dones", [])
        
        # 에피소드 완료된 환경들만 처리
        if any(dones):
            for done in dones:
                if done:
                    self.episode_count += 1
            success_rate = np.mean(self.training_env.env_method("get_success_rate"))
            self.current_success_rate = success_rate
            
            # 레벨 업 체크
            if self.episode_count >= self.min_training_epi:
                if success_rate >= self.success_threshold:
                    self.episode_count = 0
                    self.level = min(self.level + 1, self.max_level)
                    self.training_env.env_method("set_level", level=self.level)
                    
                    if self.verbose > 0:
                        print(f"Level increased to {self.level} (success rate: {success_rate:.3f})")
        
        # 주기적으로만 상세 로깅
        self.step_count += 1
        if self.step_count % self.log_freq == 0:
            self.logger.record("success_rate", self.current_success_rate)
            self.logger.record("level", self.level)
            
            # 환경 상태는 덜 자주 로깅
            if self.step_count % (self.log_freq * 5) == 0:
                front_layer_len = self.training_env.env_method("front_layer_size")
                swap_candidate_len = self.training_env.env_method("swap_candidate_size")
                reset_failed = self.training_env.env_method("get_reset_failed")
                
                self.logger.record("front_layer_size/mean", np.mean(front_layer_len))
                self.logger.record("swap_candidate_size/mean", np.mean(swap_candidate_len))
                self.logger.record("reset_failed/mean", np.mean(reset_failed))
        
        return True


In [4]:
class TrialEvalCallback(EvalCallback):
    """Callback used for evaluating and reporting a trial."""

    def __init__(
        self,
        eval_env: gym.Env,
        trial: optuna.Trial,
        n_eval_episodes: int = 5,
        eval_freq: int = 10000,
        deterministic: bool = True,
        verbose: int = 0,
    ):
        super().__init__(
            eval_env=eval_env,
            n_eval_episodes=n_eval_episodes,
            eval_freq=eval_freq,
            deterministic=deterministic,
            verbose=verbose,
        )
        self.trial = trial
        self.eval_idx = 0
        self.is_pruned = False

    def _on_step(self) -> bool:
        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            train_level = self.training_env.env_method("get_level")
            self.eval_env.env_method("set_level", level=train_level[0])
            super()._on_step()
            self.eval_idx += 1
            self.trial.report(self.last_mean_reward, self.eval_idx)

            # Prune trial if need.
            if self.trial.should_prune():
                self.is_pruned = True
                return False
        return True

In [5]:
def objective(trial: optuna.Trial) -> float:
    kwargs = DEFAULT_HYPERPARAMS.copy()
    # Sample hyperparameters.
    kwargs.update(sample_ppo_params(trial))
    # Create the RL model.
    model = PPO(**kwargs, tensorboard_log="./optuna_sb3_wog/",verbose=1, device='cuda')
    # Create env used for evaluation.
    eval_env = Monitor(gym.make(ENV_ID, start_level=1))
    # Create the callback that will periodically evaluate and report the performance.
    eval_callback = TrialEvalCallback(
        eval_env, trial, n_eval_episodes=N_EVAL_EPISODES, eval_freq=EVAL_FREQ, deterministic=True
    )

    nan_encountered = False
    try:
        print("start learning")
        model.learn(N_TIMESTEPS, callback=[eval_callback, CurriculumCallback(verbose=1)])
        print("Learning end")
    except AssertionError as e:
        # Sometimes, random hyperparams can generate NaN.
        print(e)
        nan_encountered = True
    finally:
        # Free memory.
        model.env.close()
        eval_env.close()

    # Tell the optimizer that the trial failed.
    if nan_encountered:
        return float("nan")

    if eval_callback.is_pruned:
        raise optuna.exceptions.TrialPruned()

    return eval_callback.last_mean_reward


sampler = TPESampler(n_startup_trials=N_STARTUP_TRIALS)
# Do not prune before 1/3 of the max budget is used.
pruner = MedianPruner(n_startup_trials=N_STARTUP_TRIALS, n_warmup_steps=N_EVALUATIONS // 3)

study = optuna.create_study(sampler=sampler, pruner=pruner, direction="maximize")
try:
    study.optimize(objective, n_trials=N_TRIALS, timeout= 60 * 60 * 5)  # 4 hours
except KeyboardInterrupt:
    pass

print("Number of finished trials: ", len(study.trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

print("  User attrs:")
for key, value in trial.user_attrs.items():
    print("    {}: {}".format(key, value))

[I 2025-07-01 14:31:00,604] A new study created in memory with name: no-name-5456ebbf-9b9d-410e-85c9-5306e7652fb4


Using cuda device
start learning
Logging to ./optuna_sb3_wog/PPO_1


d:\lab\circuit_route\.venv\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
d:\lab\circuit_route\.venv\Lib\site-packages\gymnasium\utils\passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be int32, actual type: float32
  logger.warn(
d:\lab\circuit_route\.venv\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logge

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1.46     |
|    ep_rew_mean     | -8.37    |
| time/              |          |
|    fps             | 209      |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 128      |
---------------------------------
---------------------------------------
| rollout/                |           |
|    ep_len_mean          | 2.01      |
|    ep_rew_mean          | -6.02     |
| time/                   |           |
|    fps                  | 237       |
|    iterations           | 2         |
|    time_elapsed         | 1         |
|    total_timesteps      | 256       |
| train/                  |           |
|    approx_kl            | 23.368805 |
|    clip_fraction        | 0.938     |
|    clip_range           | 0.2       |
|    entropy_loss         | -0.371    |
|    explained_variance   | -0.0417   |
|    learning_rate        | 0.15      |
|    loss           

[I 2025-07-01 15:27:32,173] Trial 0 finished with value: -10.15 and parameters: {'gamma': 0.0012545410597167113, 'max_grad_norm': 1.2428080568833153, 'gae_lambda': 0.05482519750239553, 'exponent_n_steps': 5, 'lr': 0.14989600275020862, 'ent_coef': 0.010594794406755653, 'vf_coef': 1.305279499835077e-05}. Best is trial 0 with value: -10.15.


Learning end
Using cuda device
start learning
Logging to ./optuna_sb3_wog/PPO_2


[W 2025-07-01 15:27:49,507] Trial 1 failed with parameters: {'gamma': 0.02011561053933403, 'max_grad_norm': 3.8993364682451883, 'gae_lambda': 0.023034257359748768, 'exponent_n_steps': 11, 'lr': 0.3623895036371654, 'ent_coef': 0.002730325486377356, 'vf_coef': 0.00915854363258381} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "d:\lab\circuit_route\.venv\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\psw04\AppData\Local\Temp\ipykernel_25040\2550026050.py", line 17, in objective
    model.learn(N_TIMESTEPS, callback=[eval_callback, CurriculumCallback(verbose=1)])
  File "d:\lab\circuit_route\.venv\Lib\site-packages\stable_baselines3\ppo\ppo.py", line 311, in learn
    return super().learn(
           ^^^^^^^^^^^^^^
  File "d:\lab\circuit_route\.venv\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py", line 324, in learn


Number of finished trials:  2
Best trial:
  Value:  -10.15
  Params: 
    gamma: 0.0012545410597167113
    max_grad_norm: 1.2428080568833153
    gae_lambda: 0.05482519750239553
    exponent_n_steps: 5
    lr: 0.14989600275020862
    ent_coef: 0.010594794406755653
    vf_coef: 1.305279499835077e-05
  User attrs:
    gamma_: 0.9987454589402833
    gae_lambda_: 0.9451748024976044
    n_steps: 32
